# 01 - Data collection (football-data.org)

Goal: pull one league (Premier League, `PL`) and check **what the free tier actually returns**:
- squads (`/competitions/PL/teams`): name, position, date of birth, nationality, contract, and possibly `marketValue`
- scorers (`/competitions/PL/scorers`): goals, assists, penalties

Run the cells in order. Raw responses are saved to `data/raw/` (git-ignored).

In [ ]:
import os, json, time
from pathlib import Path
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
TOKEN = os.getenv("FOOTBALL_DATA_TOKEN")
assert TOKEN and TOKEN != "your_token_here", "Put your API token in .env (see .env.example)"

BASE = "https://api.football-data.org/v4"
HEADERS = {"X-Auth-Token": TOKEN}
COMP = "PL"

RAW = Path("../data/raw"); RAW.mkdir(parents=True, exist_ok=True)
PROCESSED = Path("../data/processed"); PROCESSED.mkdir(parents=True, exist_ok=True)

def get(path, **params):
    r = requests.get(f"{BASE}{path}", headers=HEADERS, params=params, timeout=30)
    print(r.status_code, r.url, "| requests left this minute:", r.headers.get("X-Requests-Available-Minute"))
    r.raise_for_status()
    return r.json()

## 1. Squads

One request returns every team in the competition with its squad.

In [ ]:
teams_json = get(f"/competitions/{COMP}/teams")
(RAW / f"{COMP}_teams.json").write_text(json.dumps(teams_json, indent=2))
print(len(teams_json["teams"]), "teams")

In [ ]:
rows = []
for team in teams_json["teams"]:
    for p in team.get("squad", []):
        rows.append({
            "player_id": p.get("id"),
            "name": p.get("name"),
            "position": p.get("position"),
            "date_of_birth": p.get("dateOfBirth"),
            "nationality": p.get("nationality"),
            "contract_until": (p.get("contract") or {}).get("until"),
            "market_value": p.get("marketValue"),   # may be missing on the free tier - check below
            "team_id": team["id"],
            "team": team["name"],
        })
squads = pd.DataFrame(rows)
print(squads.shape)
squads.head()

### Check: is market value available?

This decides the data plan. If `market_value` is mostly empty, we need Transfermarkt (or another source) for the target variable.

In [ ]:
print("market_value non-null:", squads["market_value"].notna().sum(), "of", len(squads))
print("contract_until non-null:", squads["contract_until"].notna().sum(), "of", len(squads))
squads["market_value"].describe()

## 2. Top scorers (goals, assists, penalties)

In [ ]:
time.sleep(1)
# default limit is 10; ask for more. If the API ignores `limit`, the printed count will tell us.
scorers_json = get(f"/competitions/{COMP}/scorers", limit=500)
(RAW / f"{COMP}_scorers.json").write_text(json.dumps(scorers_json, indent=2))
print("scorers returned:", len(scorers_json["scorers"]))

In [ ]:
scorers = pd.DataFrame([
    {
        "player_id": s["player"]["id"],
        "name": s["player"]["name"],
        "goals": s.get("goals"),
        "assists": s.get("assists"),
        "penalties": s.get("penalties"),
        "played_matches": s.get("playedMatches"),   # may be absent depending on version/tier
    }
    for s in scorers_json["scorers"]
])
scorers.head(10)

### Note on playing time

The scorers endpoint covers players with goals/assists. Minutes played and per-90 stats are **not** in this response, so for the model we may need another source or to work with what is available (goals, assists, matches played). Decide after seeing the output above.

## 3. First merge and save

In [ ]:
players = squads.merge(
    scorers[["player_id", "goals", "assists", "penalties", "played_matches"]],
    on="player_id", how="left",
)
for col in ["goals", "assists", "penalties"]:
    players[col] = players[col].fillna(0)   # not in scorers list -> 0 goal contributions listed
players["date_of_birth"] = pd.to_datetime(players["date_of_birth"])
players["age"] = ((pd.Timestamp.today() - players["date_of_birth"]).dt.days / 365.25).round(1)

players.to_csv(PROCESSED / f"{COMP}_players_v0.csv", index=False)
print(players.shape)
players.head()

## Findings to write down (for the checkpoint presentation)

- Did the free tier return `marketValue`? How many players have it?
- How many players have goals/assists data?
- What is missing for the per-90 features, and what is the plan?